## Install Dependencies

In [ ]:
!pip install GEOparse bioservices mygene

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 258.0/258.0 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.0/57.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.3/145.3 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 720.1/720.1 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 18.6 MB/s eta 0:00:00
  Attempting uninstall: wrapt
    Found existing inst

In [ ]:
import numpy as np
import pandas as pd

from bioservices import KEGG
import re
import random
import GEOparse
from itertools import islice
from collections import defaultdict, Counter

Creating directory /root/.config/bioservices 


## GSE10072 Dataset from GEO

https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE10072

In [ ]:
# Download and parse the dataset
gse = GEOparse.get_GEO("GSE10072", destdir=".", silent=True)

In [ ]:
print(gse)

<SERIES: GSE10072 - 107 SAMPLES, 1 d(s)>


In [ ]:
# List all sample IDs
print(gse.gsms.keys())

# Inspect the first sample’s metadata
first_sample = list(gse.gsms.values())[0]
print(first_sample.metadata)

dict_keys(['GSM254625', 'GSM254626', 'GSM254627', 'GSM254628', 'GSM254629', 'GSM254630', 'GSM254631', 'GSM254632', 'GSM254633', 'GSM254634', 'GSM254635', 'GSM254636', 'GSM254637', 'GSM254638', 'GSM254639', 'GSM254640', 'GSM254641', 'GSM254642', 'GSM254643', 'GSM254644', 'GSM254645', 'GSM254646', 'GSM254647', 'GSM254648', 'GSM254649', 'GSM254650', 'GSM254651', 'GSM254652', 'GSM254653', 'GSM254654', 'GSM254655', 'GSM254656', 'GSM254657', 'GSM254658', 'GSM254659', 'GSM254660', 'GSM254661', 'GSM254662', 'GSM254663', 'GSM254664', 'GSM254665', 'GSM254666', 'GSM254667', 'GSM254668', 'GSM254669', 'GSM254670', 'GSM254671', 'GSM254672', 'GSM254673', 'GSM254674', 'GSM254675', 'GSM254676', 'GSM254677', 'GSM254678', 'GSM254679', 'GSM254680', 'GSM254681', 'GSM254682', 'GSM254683', 'GSM254684', 'GSM254685', 'GSM254686', 'GSM254687', 'GSM254688', 'GSM254689', 'GSM254690', 'GSM254691', 'GSM254692', 'GSM254693', 'GSM254694', 'GSM254695', 'GSM254696', 'GSM254697', 'GSM254698', 'GSM254699', 'GSM254700', '

In [ ]:
# Extract smoking status from characteristics_ch1 (robust to capitalization)
def extract_smoking_status(chars):
    if not chars:
        return None
    for c in chars:
        if not isinstance(c, str):
            continue
        m = re.search(r"cigarette smoking status\s*:\s*(.+)", c, flags=re.I)
        if m:
            val = m.group(1).strip()
            # Normalize common variants to the three requested strings:
            if re.search(r"never", val, flags=re.I):
                return "Never Smoked"
            if re.search(r"former", val, flags=re.I):
                return "Former Smoker"
            if re.search(r"current", val, flags=re.I):
                return "Current Smoker"
            # If value matches exactly the expected spellings just return it
            if val in ("Never Smoked", "Former Smoker", "Current Smoker"):
                return val
            return val
    return None

# Extract source_name (prefer source_name_ch1, else title)
def extract_source_name(gsm):
    src = gsm.metadata.get("source_name_ch1", []) or []
    if src:
        return src[0]
    return gsm.metadata.get("title", [""])[0] or None

# Create label for tumour (1) vs non-tumour (0)
def determine_label_from_source(source_name, title=None):
    if not source_name and not title:
        return None
    text = " ".join(filter(None, [source_name, title])).lower()
    # tumor
    if re.search(r"adenocarcinoma", text):
        return 1
    # normal/non-tumour
    if re.search(r"normal", text):
        return 0
    return None

In [ ]:
meta_rows = []
for gsm_name, gsm in gse.gsms.items():
    chars = gsm.metadata.get("characteristics_ch1", []) or []
    smoking = extract_smoking_status(chars)
    source_name = extract_source_name(gsm)
    title = gsm.metadata.get("title", [""])[0] if gsm.metadata.get("title") else ""
    label = determine_label_from_source(source_name, title)

    meta_rows.append({
        "sample_id": gsm_name,
        "source_name": source_name,
        "title": title,
        "smoking_raw_chars": chars,
        "type": smoking,
        "label": label
    })

meta_df = pd.DataFrame(meta_rows).set_index("sample_id")
print("\nMetadata summary (first 6 rows):")
print(meta_df[["source_name", "title", "type", "label"]].head(6))


Metadata summary (first 6 rows):
                          source_name                title           type  \
sample_id                                                                   
GSM254625  Adenocarcinoma of the Lung   Lung Tumor_GT00006   Never Smoked   
GSM254626          Normal Lung Tissue  Normal Lung_GT00006   Never Smoked   
GSM254627  Adenocarcinoma of the Lung   Lung Tumor_GT00007  Former Smoker   
GSM254628          Normal Lung Tissue  Normal Lung_GT00007  Former Smoker   
GSM254629  Adenocarcinoma of the Lung   Lung Tumor_GT00022   Never Smoked   
GSM254630  Adenocarcinoma of the Lung   Lung Tumor_GT00042  Former Smoker   

           label  
sample_id         
GSM254625      1  
GSM254626      0  
GSM254627      1  
GSM254628      0  
GSM254629      1  
GSM254630      1  


In [ ]:
# Print counts for smoking types and label
print(meta_df["label"].value_counts(dropna=False))

label
1    58
0    49
Name: count, dtype: int64


In [ ]:
# Check for duplicates
if not meta_df.index.is_unique:
    dup_ids = meta_df.index[meta_df.index.duplicated(keep=False)].tolist()
    print("WARNING: Duplicate sample_id(s) found in meta_df (these should be unique GSMs):", dup_ids)
else:
    print("All sample_id are unique.")

dupe_titles = meta_df["title"].value_counts()
dupe_titles = dupe_titles[dupe_titles > 1]
if not dupe_titles.empty:
    print("\nWARNING: Duplicate sample_id(s) found :")
    print(dupe_titles)
    for t in dupe_titles.index:
        members = meta_df[meta_df["title"] == t].index.tolist()
        print(f"  Title: '{t}' -> samples: {members}")
else:
    print("\nAll titles are unique.")

All sample_id are unique.

All titles are unique.


In [ ]:
# Build expression dataframe (samples x probes)
expr_df = pd.DataFrame({
    gsm_name: gsm.table.set_index("ID_REF")["VALUE"].astype(float)
    for gsm_name, gsm in gse.gsms.items()
}).transpose()

print("\nExpression dataframe shape:", expr_df.shape)


Expression dataframe shape: (107, 22283)


In [ ]:
expr_df.head()

ID_REF,1007_s_at,1053_at,117_at,121_at,1255_g_at,1294_at,1316_at,1320_at,1405_i_at,1431_at,...,AFFX-r2-Hs28SrRNA-5_at,AFFX-r2-Hs28SrRNA-M_at,AFFX-r2-P1-cre-3_at,AFFX-r2-P1-cre-5_at,AFFX-ThrX-3_at,AFFX-ThrX-5_at,AFFX-ThrX-M_at,AFFX-TrpnX-3_at,AFFX-TrpnX-5_at,AFFX-TrpnX-M_at
GSM254625,10.927084,6.895217,8.110190,9.451286,4.814477,8.774520,5.884616,6.028712,8.648566,4.648521,...,4.663081,5.558929,12.652342,13.021306,5.208449,4.966442,4.454958,4.034979,4.503507,4.396848
GSM254626,10.416978,6.924856,7.760228,9.520943,4.718640,9.368739,6.260079,6.202698,8.572660,4.812522,...,4.784434,5.655303,13.166840,13.341119,5.176357,5.054673,4.461005,4.043990,4.540112,4.464781
GSM254627,10.628538,7.550245,7.974676,9.807597,4.905163,8.407624,6.213473,6.058022,8.877887,4.946105,...,5.471763,6.513422,12.759985,12.656466,5.492227,5.400588,4.642212,4.069773,4.700017,4.714225
GSM254628,10.151180,6.699557,7.712676,9.522087,4.818076,9.000390,6.284605,6.609682,8.331652,4.901348,...,4.904406,5.578589,13.098182,13.382215,5.360600,5.141759,4.600061,3.993398,4.611192,4.579732
GSM254629,10.988512,6.826031,7.775592,9.855061,4.823958,9.104299,6.193348,6.119129,7.752896,4.968485,...,4.994150,5.429397,12.698803,13.027616,5.470905,5.151203,4.580010,4.066657,4.632961,4.633962


In [ ]:
# Add label column
aligned_meta = meta_df.reindex(expr_df.index)
label_series = aligned_meta["label"].copy()
expr_df["label"] = pd.Series(label_series.values, index=expr_df.index).astype("Int64")

print("\nExpression dataframe shape", expr_df.shape)
print(expr_df.head(5))


Expression dataframe shape (107, 22284)
ID_REF     1007_s_at   1053_at    117_at    121_at  1255_g_at   1294_at  \
GSM254625  10.927084  6.895217  8.110190  9.451286   4.814477  8.774520   
GSM254626  10.416978  6.924856  7.760228  9.520943   4.718640  9.368739   
GSM254627  10.628538  7.550245  7.974676  9.807597   4.905163  8.407624   
GSM254628  10.151180  6.699557  7.712676  9.522087   4.818076  9.000390   
GSM254629  10.988512  6.826031  7.775592  9.855061   4.823958  9.104299   

ID_REF      1316_at   1320_at  1405_i_at   1431_at  ...  \
GSM254625  5.884616  6.028712   8.648566  4.648521  ...   
GSM254626  6.260079  6.202698   8.572660  4.812522  ...   
GSM254627  6.213473  6.058022   8.877887  4.946105  ...   
GSM254628  6.284605  6.609682   8.331652  4.901348  ...   
GSM254629  6.193348  6.119129   7.752896  4.968485  ...   

ID_REF     AFFX-r2-Hs28SrRNA-M_at  AFFX-r2-P1-cre-3_at  AFFX-r2-P1-cre-5_at  \
GSM254625                5.558929            12.652342            13.02130

In [ ]:
print(expr_df["label"].value_counts(dropna=False))

label
1    58
0    49
Name: count, dtype: Int64


In [ ]:
expr_df.to_csv("original.csv", index=True)
print(f"Saved to csv.")

Saved to csv.


## Map probe -> gene symbol

In [ ]:
def fetch_gpl_table(gpl_acc="GPL96", destdir="."):
    print(f"Downloading {gpl_acc} ...")
    gpl = GEOparse.get_GEO(geo=gpl_acc, destdir=destdir, silent=True)
    table = gpl.table
    print("GPL downloaded. Table shape:", table.shape)
    return table

def build_probe2symbol_from_table(table, probe_col, symbol_col):
    mapping = {}
    missing = 0
    for idx, row in table.iterrows():
        probe = str(row[probe_col]).strip()
        symbol_raw = row.get(symbol_col, "")
        if pd.isna(symbol_raw):
            mapping[probe] = None
            missing += 1
            continue
        symbol = str(symbol_raw).strip()
        # Clean symbol: some GPLs have 'SYMBOL /// Another' or 'GENE: SYMBOL (desc)'
        # try a few heuristics:
        # 1) split on '///' or '///' or ' // '
        symbol = re.split(r"\s*///\s*|\s*//\s*|;", symbol)[0]
        # 2) if the token contains spaces and likely is description, try to extract first token that looks like a gene symbol
        tokens = re.split(r"[,\s\(\)]+", symbol)
        # keep first token that matches typical gene symbol pattern (letters, digits, dash)
        sym_candidate = None
        for t in tokens:
            if re.match(r"^[A-Za-z0-9\-\._]+$", t):
                # skip tokens that are '---' or 'NA' or numeric only
                if t.upper() in ("NA", "---"):
                    continue
                if re.search(r"[A-Za-z]", t):
                    sym_candidate = t
                    break
        if sym_candidate is None:
            sym_candidate = symbol  # fallback to raw cleaned string
        mapping[probe] = sym_candidate
    print(f"Built mapping for {len(mapping)} probes ({missing} missing symbols).")
    return mapping

In [ ]:
gpl_table = fetch_gpl_table("GPL96")
probe_col = "ID"
symbol_col = "Gene Symbol"

GPL downloaded. Table shape: (22283, 16)


In [ ]:
probe2symbol = build_probe2symbol_from_table(gpl_table, probe_col, symbol_col)

print("Example of probe2symbol: ")
example1 = dict(islice(probe2symbol.items(), 5))
example1

Built mapping for 22283 probes (1058 missing symbols).
Example of probe2symbol: 


{'1007_s_at': 'DDR1',
 '1053_at': 'RFC2',
 '117_at': 'HSPA6',
 '121_at': 'PAX8',
 '1255_g_at': 'GUCA1A'}

In [ ]:
dataset_genes = list(set(probe2symbol.values()))
print("Number of unique symbols:", len(dataset_genes))

Number of unique symbols: 13238


In [ ]:
def map_col2gene_df(df, probe2symbol, label_col):

    df = df.copy()

    label_series = df[label_col].reset_index(drop=True)

    non_probe_candidates = ["Unnamed: 0", "index", "type", label_col]
    cols_to_drop = [c for c in non_probe_candidates if c in df.columns]
    probe_df = df.drop(columns=cols_to_drop).copy()
    probe_df.columns = [str(c).strip() for c in probe_df.columns]

    # identify probe columns in the dataframe that exist in probe2symbol
    probes_present = [c for c in probe_df.columns if c in probe2symbol]
    probes_with_symbol = [p for p in probes_present if probe2symbol.get(p) is not None]

    # compose subset columns to keep
    expr = probe_df.loc[:, probes_with_symbol].apply(pd.to_numeric, errors="coerce")

    # rename probes to gene symbols
    rename_map = {probe: probe2symbol[probe] for probe in probes_with_symbol}
    expr = expr.rename(columns=rename_map)

    # collapse probes that map to the same symbol by mean
    expr = expr.groupby(expr.columns, axis=1).mean()

    # reattach label column
    final_df = pd.concat([expr.reset_index(drop=True), label_series.reset_index(drop=True)], axis=1)
    final_df = final_df.rename(columns={final_df.columns[-1]: label_col})

    return final_df

In [ ]:
gene_symbol_df = map_col2gene_df(expr_df, probe2symbol, label_col="label")

print(gene_symbol_df.shape)
gene_symbol_df.head(5)

(107, 13238)


/tmp/ipython-input-3541650562.py:24: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  expr = expr.groupby(expr.columns, axis=1).mean()


,A1CF,A2M,A4GALT,A4GNT,AAAS,AACS,AADAC,AAGAB,AAK1,AAMDC,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYX,ZZEF1,ZZZ3,abParts,label
0,6.959175,13.059984,5.231963,6.015383,8.767797,7.762807,7.118915,7.490779,6.594705,6.627443,...,6.555992,8.738558,5.550265,4.866875,8.062323,10.379627,7.152071,8.573217,11.442440,1
1,7.330048,13.702212,5.396363,6.224329,8.795224,8.001055,7.268733,7.642617,6.809936,6.928524,...,6.516934,7.372226,5.514669,4.774521,8.321107,10.476301,7.088269,8.371020,8.837601,0
2,8.036416,11.107587,5.916574,6.332004,9.060626,9.472215,7.347736,7.646721,6.936388,6.623907,...,7.415065,10.574359,6.064620,5.175124,7.901524,9.989683,7.032396,8.284845,10.014427,1
3,7.638816,13.481028,5.339658,6.231816,8.852229,7.711807,7.734262,7.407622,6.887179,7.010471,...,6.510251,7.457242,5.801066,4.896627,8.486249,10.137691,7.345402,8.365919,7.760245,0
4,8.040167,12.193454,5.370519,6.207646,9.151269,7.477610,7.083809,7.528351,6.845795,6.618909,...,6.672603,7.756057,5.887176,5.149239,8.179661,9.907049,6.964879,8.314202,10.409890,1


In [ ]:
gene_symbol_df.to_csv("original_mapped.csv", index=True)
print("saved csv")

saved csv


In [ ]:
gene_symbols = {c for c in gene_symbol_df.columns if c != "label"}

## Get genes from KEGG cancer pathways

In [ ]:
unique_intersect_gene = set()

In [ ]:
def fetch_kegg_aml_symbols(pathway):
    k = KEGG()
    entry = k.get(pathway)
    parsed = k.parse(entry)
    symbols = set()
    if 'GENE' in parsed:
        for entrez, desc in parsed['GENE'].items():
            m = re.match(r"^([A-Za-z0-9\-\._]+)", desc)
            if m:
                symbols.add(m.group(1).upper().strip())
    return symbols

def add_to_set(list):
    for item in list:
      if item not in unique_intersect_gene:
        unique_intersect_gene.add(item)
      pass

def get_intersect_genes(gene_symbols, pathway):
    symbols_in_pathway = fetch_kegg_aml_symbols(pathway)
    print(f'Number of symbols in {pathway}: {len(symbols_in_pathway)}')

    intersected_genes = sorted(list(gene_symbols.intersection(symbols_in_pathway)))
    print("Matched genes:", len(intersected_genes))

    add_to_set(intersected_genes)
    print("Number of intersected gene: ", len(unique_intersect_gene))

In [ ]:
kegg_pathway_list = [
    'hsa05200',
    'hsa05205',
    'hsa05206',
    'hsa05202',
    'hsa05223',
    'hsa05230',
    'hsa05222',
    'hsa05224',
    'hsa05215',
    'hsa05213',
    'hsa05226',
    'hsa05210',
    'hsa05212',
    'hsa05216',
    'hsa05219',
    'hsa05231',
    'hsa05235',
    'hsa05217',
    'hsa05208',
    'hsa05203',
    'hsa05169',
    'hsa05211',
    'hsa05204',
    'hsa05225',
    'hsa05207',
    'hsa05218',
    'hsa05214',
    'hsa05165',
    'hsa04371',
    'hsa03460',
    'hsa03320'
]

In [ ]:
for i in kegg_pathway_list:
  get_intersect_genes(gene_symbols, i)

Creating directory /root/.cache/bioservices 
Welcome to Bioservices
It looks like you do not have a configuration file.
We are creating one with default values in /root/.config/bioservices/bioservices.cfg .
Done
Number of symbols in hsa05200: 533
Matched genes: 488
Number of intersected gene:  488
Number of symbols in hsa05205: 204
Matched genes: 187
Number of intersected gene:  571
Number of symbols in hsa05206: 320
Matched genes: 149
Number of intersected gene:  632
Number of symbols in hsa05202: 201
Matched genes: 166
Number of intersected gene:  738
Number of symbols in hsa05223: 73
Matched genes: 70
Number of intersected gene:  743
Number of symbols in hsa05230: 71
Matched genes: 66
Number of intersected gene:  772
Number of symbols in hsa05222: 93
Matched genes: 91
Number of intersected gene:  772
Number of symbols in hsa05224: 148
Matched genes: 132
Number of intersected gene:  777
Number of symbols in hsa05215: 106
Matched genes: 101
Number of intersected gene:  795
Number of s

In [ ]:
with open("kegg_intersect.txt", "w") as f:
  f.write(repr(unique_intersect_gene))

## Extract subset from intersected genes

In [ ]:
df = pd.read_csv('original_mapped.csv')
df.head(5)

,Unnamed: 0,A1CF,A2M,A4GALT,A4GNT,AAAS,AACS,AADAC,AAGAB,AAK1,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYX,ZZEF1,ZZZ3,abParts,label
0,0,6.959175,13.059984,5.231963,6.015383,8.767797,7.762807,7.118915,7.490779,6.594705,...,6.555992,8.738558,5.550265,4.866875,8.062323,10.379627,7.152071,8.573217,11.442440,1
1,1,7.330048,13.702212,5.396363,6.224329,8.795224,8.001055,7.268733,7.642617,6.809936,...,6.516934,7.372226,5.514669,4.774521,8.321107,10.476301,7.088269,8.371020,8.837601,0
2,2,8.036416,11.107587,5.916574,6.332004,9.060626,9.472215,7.347736,7.646721,6.936388,...,7.415065,10.574359,6.064620,5.175124,7.901524,9.989683,7.032396,8.284845,10.014427,1
3,3,7.638816,13.481028,5.339658,6.231816,8.852229,7.711807,7.734262,7.407622,6.887179,...,6.510251,7.457242,5.801066,4.896627,8.486249,10.137691,7.345402,8.365919,7.760245,0
4,4,8.040167,12.193454,5.370519,6.207646,9.151269,7.477610,7.083809,7.528351,6.845795,...,6.672603,7.756057,5.887176,5.149239,8.179661,9.907049,6.964879,8.314202,10.409890,1


In [ ]:
with open("kegg_intersect.txt") as f:
    initial_intersect = eval(f.read())

print(len(initial_intersect))
print(list(initial_intersect)[:10])

1431
['PPP3R1', 'LRP5', 'VIM', 'GRM1', 'NFE2L2', 'JAG2', 'PRKCI', 'PAX5', 'MMP9', 'IL7R']


In [ ]:
df.columns = [str(c).strip() for c in df.columns]
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

In [ ]:
df.head()

,A1CF,A2M,A4GALT,A4GNT,AAAS,AACS,AADAC,AAGAB,AAK1,AAMDC,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYX,ZZEF1,ZZZ3,abParts,label
0,6.959175,13.059984,5.231963,6.015383,8.767797,7.762807,7.118915,7.490779,6.594705,6.627443,...,6.555992,8.738558,5.550265,4.866875,8.062323,10.379627,7.152071,8.573217,11.442440,1
1,7.330048,13.702212,5.396363,6.224329,8.795224,8.001055,7.268733,7.642617,6.809936,6.928524,...,6.516934,7.372226,5.514669,4.774521,8.321107,10.476301,7.088269,8.371020,8.837601,0
2,8.036416,11.107587,5.916574,6.332004,9.060626,9.472215,7.347736,7.646721,6.936388,6.623907,...,7.415065,10.574359,6.064620,5.175124,7.901524,9.989683,7.032396,8.284845,10.014427,1
3,7.638816,13.481028,5.339658,6.231816,8.852229,7.711807,7.734262,7.407622,6.887179,7.010471,...,6.510251,7.457242,5.801066,4.896627,8.486249,10.137691,7.345402,8.365919,7.760245,0
4,8.040167,12.193454,5.370519,6.207646,9.151269,7.477610,7.083809,7.528351,6.845795,6.618909,...,6.672603,7.756057,5.887176,5.149239,8.179661,9.907049,6.964879,8.314202,10.409890,1


In [ ]:
# keep only intersected genes that are present in the df
genes_present = [g for g in initial_intersect if g in df.columns]
missing_genes = [g for g in initial_intersect if g not in df.columns]
cols_to_keep = genes_present + ["label"]

print(f"Intersected genes found in dataframe: {len(genes_present)}")

Intersected genes found in dataframe: 1431


In [ ]:
reduced_df = df.loc[:, cols_to_keep].copy()

print("Reduced dataframe shape (rows x cols):", reduced_df.shape)
print("Label distribution in reduced set:")
print(reduced_df["label"].value_counts())

Reduced dataframe shape (rows x cols): (107, 1432)
Label distribution in reduced set:
label
1    58
0    49
Name: count, dtype: int64


In [ ]:
reduced_df.to_csv("reduced.csv", index=False)
print(f"Saved reduced dataframe")

Saved reduced dataframe
